# OneBill → Dynamics 365 Sales (Dataverse) Customer Sync

This notebook performs a **one-off sync** of customer records from OneBill into the Dataverse `account` table.

## What it does

1. Pages through OneBill's `/customers` endpoint, pulling every customer record.
2. Maps each OneBill customer to the Dataverse `account` schema.
3. **Upserts** each account into Dataverse using `accountnumber` as the match key — existing accounts are updated, new ones are created.

## Before you run

Two pieces of one-time setup are required in your Dataverse environment:

1. **Alternate key on `accountnumber`** — under Power Apps → Tables → Account → Keys, create a new key on the `Account Number` column. The upsert pattern used here (`PATCH /accounts(accountnumber='...')`) requires this key to exist, otherwise Dataverse returns 404.
2. **Application User** — your Azure AD app registration must be added as an Application User inside the Dataverse environment, with a security role permitting create/write on the Account table.

## How to use

Run the cells in order. The sync itself is gated behind a `DRY_RUN` flag near the bottom — leave it on for the first pass to confirm the mapping looks right, then flip it off to actually write to Dataverse.

## 1. Install dependencies

Two packages are needed:

- **`requests`** — HTTP client for both OneBill and the Dataverse Web API.
- **`msal`** — Microsoft's auth library, used to acquire an access token via the client-credentials flow.

Run this once per environment. Skip if you've already installed them.

In [ ]:
%pip install requests msal

## 2. Imports and logging

Standard library imports plus the two installed dependencies. Logging is configured to print timestamped INFO-level messages so you can follow the sync's progress.

In [ ]:
from __future__ import annotations

import logging
import os
import time
from dataclasses import dataclass
from typing import Any, Iterator

import requests
import msal

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("sync")

## 3. Configuration

Fill in your credentials below — or, preferably, set them as environment variables and let the cell read from there. Keeping secrets out of the notebook file is safer if you share or version-control it.

**OneBill**
- `ONEBILL_BASE_URL` — your OneBill API root (default is the public hostname; swap if your tenant uses a custom domain).
- `ONEBILL_API_KEY` — bearer token from your OneBill tenant.

**Dataverse**
- `DATAVERSE_URL` — your environment URL, e.g. `https://contoso.crm.dynamics.com`.
- `AZURE_TENANT_ID` — the GUID of your Azure AD tenant.
- `AZURE_CLIENT_ID` / `AZURE_CLIENT_SECRET` — credentials from the app registration you set up for this integration.

In [ ]:
# OneBill
ONEBILL_BASE_URL = os.getenv("ONEBILL_BASE_URL", "https://api.onebillsoftware.com")
ONEBILL_API_KEY  = os.getenv("ONEBILL_API_KEY",  "<your-onebill-api-key>")

# Dataverse / Dynamics 365 Sales
DATAVERSE_URL       = os.getenv("DATAVERSE_URL",       "https://<your-org>.crm.dynamics.com")
AZURE_TENANT_ID     = os.getenv("AZURE_TENANT_ID",     "<tenant-guid>")
AZURE_CLIENT_ID     = os.getenv("AZURE_CLIENT_ID",     "<app-registration-client-id>")
AZURE_CLIENT_SECRET = os.getenv("AZURE_CLIENT_SECRET", "<app-registration-secret>")

# Throttle — Dataverse limits are generous, but a small pause keeps us well under them.
SLEEP_BETWEEN_UPSERTS = 0.05  # seconds

## 4. OneBill client

A minimal REST client that pages through OneBill's customers endpoint.

**Things you may need to tweak per tenant:**
- **Auth header** — most OneBill installs use `Authorization: Bearer <token>`, but some use `X-API-KEY`. Adjust `_headers` if your tenant differs.
- **Response shape** — OneBill API versions wrap records under different keys (`data`, `customers`, `results`, or a bare array). The iterator tries each in turn.
- **Pagination** — we use `page` / `pageSize` query params and stop when we receive a short page.

In [ ]:
class OneBillClient:
    """Minimal OneBill REST client for the Customers endpoint."""

    def __init__(self, base_url: str, api_key: str):
        self.base_url = base_url.rstrip("/")
        self.api_key = api_key
        self.session = requests.Session()

    def _headers(self) -> dict[str, str]:
        return {
            "Authorization": f"Bearer {self.api_key}",
            "Accept": "application/json",
        }

    def iter_customers(self, page_size: int = 100) -> Iterator[dict[str, Any]]:
        """Yield customer records, paging through the API until exhausted."""
        page = 1
        while True:
            resp = self.session.get(
                f"{self.base_url}/api/v1/customers",
                headers=self._headers(),
                params={"page": page, "pageSize": page_size},
                timeout=30,
            )
            resp.raise_for_status()
            payload = resp.json()

            # OneBill responses vary by version; try common shapes.
            records = (
                payload.get("data")
                or payload.get("customers")
                or payload.get("results")
                or (payload if isinstance(payload, list) else [])
            )
            if not records:
                return

            yield from records

            # A short page means we've hit the end.
            if len(records) < page_size:
                return
            page += 1

## 5. Dataverse client

Talks to the Dataverse Web API (`/api/data/v9.2`).

**Auth** uses MSAL's client-credentials flow: the app registration authenticates as itself (no user), trades its client secret for an access token, and uses that token as a bearer credential on every request. The scope must be `<environment-url>/.default`.

**Upsert via alternate key** is the key trick here. By PATCHing to a URL of the form `/accounts(accountnumber='ABC123')`, Dataverse will create the account if it doesn't exist and update it if it does — atomically, on the server side. This is why the alternate key setup is mandatory.

In [ ]:
class DataverseClient:
    """Dataverse Web API client using MSAL client-credentials flow."""

    def __init__(self, base_url: str, tenant_id: str, client_id: str, client_secret: str):
        self.base_url = base_url.rstrip("/")
        self.api_url = f"{self.base_url}/api/data/v9.2"
        self._token = self._acquire_token(tenant_id, client_id, client_secret)
        self.session = requests.Session()

    def _acquire_token(self, tenant_id: str, client_id: str, client_secret: str) -> str:
        """Exchange client credentials for a Dataverse-scoped access token."""
        app = msal.ConfidentialClientApplication(
            client_id=client_id,
            client_credential=client_secret,
            authority=f"https://login.microsoftonline.com/{tenant_id}",
        )
        # Scope must be the Dataverse environment URL + /.default
        result = app.acquire_token_for_client(scopes=[f"{self.base_url}/.default"])
        if "access_token" not in result:
            raise RuntimeError(f"Token acquisition failed: {result.get('error_description')}")
        return result["access_token"]

    def _headers(self, extra: dict[str, str] | None = None) -> dict[str, str]:
        h = {
            "Authorization": f"Bearer {self._token}",
            "Accept": "application/json",
            "OData-MaxVersion": "4.0",
            "OData-Version": "4.0",
            "Content-Type": "application/json; charset=utf-8",
        }
        if extra:
            h.update(extra)
        return h

    def upsert_account_by_accountnumber(self, account_number: str, attrs: dict[str, Any]) -> str:
        """Upsert an account using the `accountnumber` alternate key.

        Returns "created" if a new row was inserted, "updated" if an existing
        row was patched.
        """
        # Escape any single quotes in the key value per OData rules.
        safe = account_number.replace("'", "''")
        url = f"{self.api_url}/accounts(accountnumber='{safe}')"

        resp = self.session.patch(
            url,
            headers=self._headers({"Prefer": "return=representation"}),
            json=attrs,
            timeout=30,
        )

        if resp.status_code in (200, 201, 204):
            # 201 = created, 200/204 = updated. Some configs return 204 for both.
            return "created" if resp.status_code == 201 else "updated"

        # Surface the server's error message for easier debugging.
        try:
            err = resp.json().get("error", {}).get("message", resp.text)
        except Exception:
            err = resp.text
        raise RuntimeError(f"Upsert failed ({resp.status_code}) for {account_number}: {err}")

## 6. Field mapping

Translates a OneBill customer record into the Dataverse `account` schema.

**This is the cell you're most likely to edit.** OneBill field names vary between tenants and API versions, so the mapper tries a handful of common variants on each lookup (`companyName` vs `name`, `billingAddress` vs `address`, etc.). After your first dry run, inspect a sample customer and trim the mapping down to the fields your tenant actually returns.

Optional fields are only included when present — this avoids overwriting good Dataverse data with nulls from OneBill.

In [ ]:
@dataclass
class MappedAccount:
    account_number: str
    attrs: dict[str, Any]


def map_customer_to_account(customer: dict[str, Any]) -> MappedAccount | None:
    """Translate a OneBill customer record into Dataverse `account` attributes.

    Returns None if the customer has no usable identifier.
    """
    account_number = (
        customer.get("customerId")
        or customer.get("customer_id")
        or customer.get("id")
    )
    if not account_number:
        return None

    name = (
        customer.get("companyName")
        or customer.get("name")
        or f"{customer.get('firstName', '')} {customer.get('lastName', '')}".strip()
        or f"OneBill customer {account_number}"
    )

    attrs: dict[str, Any] = {
        "name": name,
        "accountnumber": str(account_number),
    }

    # Top-level contact fields — only set when present.
    if email := customer.get("email") or customer.get("emailAddress"):
        attrs["emailaddress1"] = email
    if phone := customer.get("phone") or customer.get("phoneNumber"):
        attrs["telephone1"] = phone
    if website := customer.get("website"):
        attrs["websiteurl"] = website

    # Billing address — OneBill commonly nests this under `billingAddress`.
    addr = customer.get("billingAddress") or customer.get("address") or {}
    if addr:
        if v := addr.get("line1") or addr.get("street1"):
            attrs["address1_line1"] = v
        if v := addr.get("line2") or addr.get("street2"):
            attrs["address1_line2"] = v
        if v := addr.get("city"):
            attrs["address1_city"] = v
        if v := addr.get("state") or addr.get("region"):
            attrs["address1_stateorprovince"] = v
        if v := addr.get("postalCode") or addr.get("zip"):
            attrs["address1_postalcode"] = v
        if v := addr.get("country"):
            attrs["address1_country"] = v

    return MappedAccount(account_number=str(account_number), attrs=attrs)

## 7. Dry-run preview

Before writing anything, pull a handful of customers from OneBill and print the mapped output. This is your chance to verify that:

- Authentication to OneBill works.
- The response shape matches what `OneBillClient.iter_customers` expects.
- The mapper picks up the field names your tenant actually returns.

Adjust the `PREVIEW_LIMIT` if you want to see more samples.

In [ ]:
PREVIEW_LIMIT = 5

onebill = OneBillClient(ONEBILL_BASE_URL, ONEBILL_API_KEY)

log.info("Fetching up to %d customers from OneBill for preview...", PREVIEW_LIMIT)
for i, customer in enumerate(onebill.iter_customers(), start=1):
    if i > PREVIEW_LIMIT:
        break
    mapped = map_customer_to_account(customer)
    if mapped is None:
        log.warning("Skipping customer with no ID: %r", customer)
        continue
    print(f"\n--- Customer {i} ---")
    print(f"account_number: {mapped.account_number}")
    for key, value in mapped.attrs.items():
        print(f"  {key}: {value}")

## 8. Authenticate to Dataverse

Acquire the access token and verify the connection. If this cell fails, double-check that:

- The app registration has the **Dynamics CRM `user_impersonation`** API permission granted with admin consent.
- The app registration has been added as an **Application User** inside the Dataverse environment (Power Platform Admin Center → Environments → Settings → Users + permissions → Application users).
- The Application User has a security role that includes `Create` and `Write` on the Account table.

In [ ]:
log.info("Authenticating to Dataverse at %s", DATAVERSE_URL)
dataverse = DataverseClient(
    DATAVERSE_URL, AZURE_TENANT_ID, AZURE_CLIENT_ID, AZURE_CLIENT_SECRET,
)
log.info("Token acquired \u2014 ready to write.")

## 9. Run the sync

This is the cell that actually writes to Dataverse.

**Settings:**
- `DRY_RUN` — when `True`, the cell logs what *would* happen without making any Dataverse calls. Leave this on for your first run.
- `LIMIT` — optional cap on the number of records processed (set to `None` for the full set). Useful for a controlled small-batch test before going wide.

**Output:** a per-record log line showing `created` / `updated` / `failed`, followed by a summary at the end.

In [ ]:
DRY_RUN = True   # Flip to False once the preview looks correct.
LIMIT   = 10     # Set to None to process everything.

stats = {"created": 0, "updated": 0, "skipped": 0, "failed": 0}

for i, customer in enumerate(onebill.iter_customers(), start=1):
    if LIMIT is not None and i > LIMIT:
        log.info("Reached LIMIT=%d, stopping.", LIMIT)
        break

    mapped = map_customer_to_account(customer)
    if mapped is None:
        log.warning("Skipping customer with no ID: %r", customer)
        stats["skipped"] += 1
        continue

    if DRY_RUN:
        log.info("[dry-run] would upsert account_number=%s name=%r",
                 mapped.account_number, mapped.attrs.get("name"))
        stats["updated"] += 1  # bucket dry-run hits here for the summary
        continue

    try:
        result = dataverse.upsert_account_by_accountnumber(mapped.account_number, mapped.attrs)
        stats[result] += 1
        log.info("%-7s  %s  %s", result, mapped.account_number, mapped.attrs.get("name"))
    except Exception as e:
        stats["failed"] += 1
        log.error("FAILED  %s  %s", mapped.account_number, e)

    time.sleep(SLEEP_BETWEEN_UPSERTS)

log.info("Done. created=%(created)d updated=%(updated)d skipped=%(skipped)d failed=%(failed)d", stats)

## 10. Next steps

Once a dry run looks correct and a small `LIMIT`ed real run succeeds:

1. Set `DRY_RUN = False` and `LIMIT = None` in the cell above and re-run.
2. Spot-check a few accounts in Dynamics to confirm the field values landed where you expected.
3. If you want this on a schedule rather than as a one-off, the same client classes drop straight into an Azure Function or a cron-driven container — just lift cells 2–6 and 8–9 into a `.py` file and trigger `main()` on whatever cadence you want.